# Data Preparation and Master Analytical Dataset

## DATA 698 Master's Research Capstone

**Project:** *When Severity Does Not Equal Risk: A Longitudinal Analysis of Vulnerability Severity and Real-World Exploitation*

### Purpose

This notebook creates the authoritative analytical dataset used for the capstone analysis.

The workflow is:

**NVD 2020–2026 → Clean and standardize → One row per CVE → Join CISA KEV → Join EPSS → Create threshold and mismatch features → Run data QA → Export processed dataset**

Unlike `feasibility_eda.ipynb`, which was used to establish project feasibility and conduct preliminary exploratory analysis, this notebook focuses specifically on reproducible data preparation.

### Output

The final dataset will contain one row per CVE and will be exported to:

`data/processed/master_vulnerability_dataset.parquet`

This processed dataset will serve as the primary input for subsequent exploratory analysis, statistical modeling, and visualization.

In [41]:
# Imports and project paths

import json
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 60)
pd.set_option("display.max_rows", 100)


# Locate the DATA 698 capstone project root

cwd = Path.cwd().resolve()

if (cwd / "data").exists() and (cwd / "notebooks").exists():
    PROJECT_ROOT = cwd
elif cwd.name == "notebooks" and (cwd.parent / "data").exists():
    PROJECT_ROOT = cwd.parent
else:
    raise FileNotFoundError(
        "Could not locate the data698-capstone project root."
    )

# Project directories
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
FIGURES_DIR = PROJECT_ROOT / "figures"
RESULTS_DIR = PROJECT_ROOT / "results"

# Create output directories if necessary
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Analysis years
YEARS = list(range(2020, 2027))

# Raw source files
KEV_FILE = RAW_DATA_DIR / "known_exploited_vulnerabilities.csv"
EPSS_FILE = RAW_DATA_DIR / "epss_scores-2026-09-02.csv.gz"

# Final processed dataset
MASTER_DATA_FILE = (
    PROCESSED_DIR / "master_vulnerability_dataset.csv"
)

# Verify paths
print("Project root:", PROJECT_ROOT)
print("Raw data:", RAW_DATA_DIR)
print("Processed data:", PROCESSED_DIR)
print("Figures:", FIGURES_DIR)
print("Results:", RESULTS_DIR)

print("\nKEV file exists:", KEV_FILE.exists())
print("EPSS file exists:", EPSS_FILE.exists())

Project root: /Users/aaliyahmjh/Documents/data698-capstone
Raw data: /Users/aaliyahmjh/Documents/data698-capstone/data/raw
Processed data: /Users/aaliyahmjh/Documents/data698-capstone/data/processed
Figures: /Users/aaliyahmjh/Documents/data698-capstone/figures
Results: /Users/aaliyahmjh/Documents/data698-capstone/results

KEV file exists: True
EPSS file exists: True


## Data Preparation Workflow

This notebook is organized into the following stages:

1. **Validate source files**
   - Confirm NVD files for 2020–2026
   - Confirm CISA KEV file
   - Confirm EPSS snapshot

2. **Load and standardize NVD data**
   - Extract CVE identifiers and publication dates
   - Extract CVSS scores, severity, versions, and attack characteristics
   - Extract CWE, vendor, and product information

3. **Create one-row-per-CVE dataset**
   - Check CVE uniqueness
   - Resolve duplicate records if necessary
   - Retain publication year separately from source-file year

4. **Join CISA KEV**
   - Create known-exploitation indicator
   - Retain relevant KEV metadata

5. **Join EPSS**
   - Add current EPSS score and percentile

6. **Create analysis features**
   - CVSS threshold indicators
   - KEV mismatch indicators

7. **Data quality assurance**
   - Row and CVE counts
   - Duplicate checks
   - CVSS coverage
   - CVSS v3.x coverage
   - KEV and EPSS match rates
   - Missingness
   - Publication-year validation

8. **Export master analytical dataset**
   - Save the validated dataset for downstream analysis

## 1. Validate Source Files

Before constructing the master analytical dataset, I verified that all required raw data files are available.


This validation step ensures that the data preparation pipeline does not continue with missing source data.

In [2]:
# Locate the NVD ZIP file for each analysis year

def find_nvd_zip(year):
    matches = sorted(RAW_DATA_DIR.glob(f"*{year}*.json.zip"))

    if not matches:
        raise FileNotFoundError(
            f"No NVD 2.0 ZIP found for {year} in {RAW_DATA_DIR.resolve()}"
        )

    if len(matches) > 1:
        print(f"Warning: multiple NVD files found for {year}:")
        for match in matches:
            print("  ", match.name)

    return matches[0]


# Validate all required source files

source_validation = []

for year in YEARS:
    try:
        nvd_file = find_nvd_zip(year)

        source_validation.append({
            "source": "NVD",
            "year": year,
            "file": nvd_file.name,
            "exists": True
        })

    except FileNotFoundError:
        source_validation.append({
            "source": "NVD",
            "year": year,
            "file": None,
            "exists": False
        })


source_validation.extend([
    {
        "source": "CISA KEV",
        "year": None,
        "file": KEV_FILE.name,
        "exists": KEV_FILE.exists()
    },
    {
        "source": "EPSS",
        "year": None,
        "file": EPSS_FILE.name,
        "exists": EPSS_FILE.exists()
    }
])

source_validation_df = pd.DataFrame(source_validation)

display(source_validation_df)

,source,year,file,exists
0,NVD,2020.0,nvdcve-2.0-2020.json.zip,True
1,NVD,2021.0,nvdcve-2.0-2021.json.zip,True
2,NVD,2022.0,nvdcve-2.0-2022.json.zip,True
3,NVD,2023.0,nvdcve-2.0-2023.json.zip,True
4,NVD,2024.0,nvdcve-2.0-2024.json.zip,True
5,NVD,2025.0,nvdcve-2.0-2025.json.zip,True
6,NVD,2026.0,nvdcve-2.0-2026.json.zip,True
7,CISA KEV,NaN,known_exploited_vulnerabilities.csv,True
8,EPSS,NaN,epss_scores-2026-09-02.csv.gz,True


In [3]:
# Stop the pipeline if any required source file is missing

missing_sources = source_validation_df[
    ~source_validation_df["exists"]
]

if not missing_sources.empty:
    raise FileNotFoundError(
        "One or more required source files are missing."
    )

print("All required source files are available.")

All required source files are available.


## 2. Load and Standardize NVD Data

This section loads the NVD JSON 2.0 files for 2020–2026 and converts the nested vulnerability records into a standardized tabular format.

For each CVE, the dataset retains:

- CVE identifier and publication information
- CVSS score, severity, and version
- CVSS attack characteristics
- CWE classification
- Vendor and product information
- Source-file year, CVE year, and publication year

When multiple CVSS metric versions are available, the extraction logic uses the following preference order:

**CVSS v3.1 → CVSS v3.0 → CVSS v4.0 → CVSS v2**

The CVSS version is retained so later analyses can distinguish between scoring-system versions. Analyses involving comparable CVSS attack characteristics will primarily use compatible CVSS v3.x records.

In [4]:
# Helper functions for NVD extraction


def first_english_description(descriptions):
    """Return the first English-language description."""
    for item in descriptions or []:
        if item.get("lang") == "en":
            return item.get("value")
    return None


def extract_cwes(weaknesses):
    """Extract unique CWE identifiers from an NVD record."""
    cwes = []

    for weakness in weaknesses or []:
        for description in weakness.get("description", []):
            value = description.get("value")

            if value and value.startswith("CWE-"):
                cwes.append(value)

    # Remove duplicates while preserving order
    return list(dict.fromkeys(cwes))


def extract_vendor_products(configurations):
    """
    Extract unique vendor and product names from vulnerable CPE entries.
    """
    vendors = []
    products = []

    def process_nodes(nodes):
        for node in nodes or []:

            for cpe_match in node.get("cpeMatch", []):
                if not cpe_match.get("vulnerable", False):
                    continue

                criteria = cpe_match.get("criteria", "")

                # CPE 2.3 format:
                # cpe:2.3:<part>:<vendor>:<product>:...
                parts = criteria.split(":")

                if len(parts) >= 5:
                    vendor = parts[3]
                    product = parts[4]

                    if vendor and vendor != "*":
                        vendors.append(vendor)

                    if product and product != "*":
                        products.append(product)

            # Handle nested nodes if present
            process_nodes(node.get("nodes", []))

    process_nodes(configurations)

    vendors = list(dict.fromkeys(vendors))
    products = list(dict.fromkeys(products))

    return vendors, products

In [5]:
def extract_cvss(metrics):
    """
    Extract the preferred CVSS metric from an NVD vulnerability record.

    Preference:
    v3.1 -> v3.0 -> v4.0 -> v2
    """

    metric_preferences = [
        ("cvssMetricV31", "3.1"),
        ("cvssMetricV30", "3.0"),
        ("cvssMetricV40", "4.0"),
        ("cvssMetricV2", "2.0"),
    ]

    for metric_key, version_label in metric_preferences:

        metric_list = metrics.get(metric_key, [])

        if not metric_list:
            continue

        # Prefer a Primary NVD metric when available
        primary_metrics = [
            m for m in metric_list
            if m.get("type") == "Primary"
        ]

        metric = primary_metrics[0] if primary_metrics else metric_list[0]

        cvss_data = metric.get("cvssData", {})

        return {
            "cvss_version": cvss_data.get("version", version_label),
            "cvss_source": metric.get("source"),
            "cvss_type": metric.get("type"),
            "cvss_score": cvss_data.get("baseScore"),
            "cvss_severity": (
                cvss_data.get("baseSeverity")
                or metric.get("baseSeverity")
            ),
            "attack_vector": cvss_data.get("attackVector"),
            "attack_complexity": cvss_data.get("attackComplexity"),
            "privileges_required": cvss_data.get("privilegesRequired"),
            "user_interaction": cvss_data.get("userInteraction"),
            "confidentiality_impact": cvss_data.get("confidentialityImpact"),
            "integrity_impact": cvss_data.get("integrityImpact"),
            "availability_impact": cvss_data.get("availabilityImpact"),
        }

    # No usable CVSS metric
    return {
        "cvss_version": None,
        "cvss_source": None,
        "cvss_type": None,
        "cvss_score": None,
        "cvss_severity": None,
        "attack_vector": None,
        "attack_complexity": None,
        "privileges_required": None,
        "user_interaction": None,
        "confidentiality_impact": None,
        "integrity_impact": None,
        "availability_impact": None,
    }

In [6]:
def parse_nvd_record(vulnerability, source_file_year):
    """Convert one NVD vulnerability record into one flat row."""

    cve_data = vulnerability.get("cve", {})

    cve_id = cve_data.get("id")

    published = pd.to_datetime(
        cve_data.get("published"),
        errors="coerce",
        utc=True
    )

    last_modified = pd.to_datetime(
        cve_data.get("lastModified"),
        errors="coerce",
        utc=True
    )

    # CWE information
    cwes = extract_cwes(cve_data.get("weaknesses", []))

    # Vendor/product information
    vendors, products = extract_vendor_products(
        cve_data.get("configurations", [])
    )

    # CVSS information
    cvss = extract_cvss(
        cve_data.get("metrics", {})
    )

    # Extract year from CVE identifier
    try:
        cve_year = int(cve_id.split("-")[1])
    except (AttributeError, IndexError, ValueError):
        cve_year = None

    row = {
        "cve": cve_id,
        "source_file_year": source_file_year,
        "published": published,
        "last_modified": last_modified,
        "vuln_status": cve_data.get("vulnStatus"),

        "cwe": cwes[0] if cwes else None,
        "all_cwes": cwes,

        "vendor": vendors[0] if vendors else None,
        "product": products[0] if products else None,
        "all_vendors": vendors,
        "all_products": products,

        "publication_year": (
            published.year
            if not pd.isna(published)
            else None
        ),

        "cve_year": cve_year,
    }

    row.update(cvss)

    return row

In [7]:
# Load and parse NVD data for 2020–2026

nvd_rows = []

for year in YEARS:

    nvd_file = find_nvd_zip(year)

    print(f"Loading {year}: {nvd_file.name}")

    with zipfile.ZipFile(nvd_file, "r") as z:

        json_files = [
            name for name in z.namelist()
            if name.endswith(".json")
        ]

        if not json_files:
            raise ValueError(
                f"No JSON file found inside {nvd_file.name}"
            )

        with z.open(json_files[0]) as f:
            nvd_json = json.load(f)

    vulnerabilities = nvd_json.get("vulnerabilities", [])

    print(f"  {len(vulnerabilities):,} records")

    for vulnerability in vulnerabilities:
        nvd_rows.append(
            parse_nvd_record(
                vulnerability,
                source_file_year=year
            )
        )


nvd_df = pd.DataFrame(nvd_rows)

print("\nCombined NVD records:", f"{len(nvd_df):,}")
print("Columns:", len(nvd_df.columns))

Loading 2020: nvdcve-2.0-2020.json.zip
  21,074 records
Loading 2021: nvdcve-2.0-2021.json.zip
  23,461 records
Loading 2022: nvdcve-2.0-2022.json.zip
  27,538 records
Loading 2023: nvdcve-2.0-2023.json.zip
  31,401 records
Loading 2024: nvdcve-2.0-2024.json.zip
  39,233 records
Loading 2025: nvdcve-2.0-2025.json.zip
  45,212 records
Loading 2026: nvdcve-2.0-2026.json.zip
  52,023 records

Combined NVD records: 239,942
Columns: 25


In [8]:
# Inspect standardized NVD dataframe

print("Shape:", nvd_df.shape)

display(nvd_df.head())

print("\nColumn names:")
for column in nvd_df.columns:
    print("-", column)

Shape: (239942, 25)


,cve,source_file_year,published,last_modified,vuln_status,cwe,all_cwes,vendor,product,all_vendors,all_products,publication_year,cve_year,cvss_version,cvss_source,cvss_type,cvss_score,cvss_severity,attack_vector,attack_complexity,privileges_required,user_interaction,confidentiality_impact,integrity_impact,availability_impact
0,CVE-2020-5179,2020,2020-01-02 14:16:37.097000+00:00,2026-06-17 03:20:59+00:00,Modified,CWE-78,[CWE-78],comtech,stampede_fx-1010_firmware,[comtech],[stampede_fx-1010_firmware],2020,2020,3.1,nvd@nist.gov,Primary,7.2,HIGH,NETWORK,LOW,HIGH,NONE,HIGH,HIGH,HIGH
1,CVE-2020-5310,2020,2020-01-03 01:15:11.087000+00:00,2026-06-17 03:21:15.233000+00:00,Modified,CWE-190,[CWE-190],python,pillow,"[python, canonical, fedoraproject]","[pillow, ubuntu_linux, fedora]",2020,2020,3.1,nvd@nist.gov,Primary,8.8,HIGH,NETWORK,LOW,NONE,REQUIRED,HIGH,HIGH,HIGH
2,CVE-2020-5311,2020,2020-01-03 01:15:11.167000+00:00,2026-06-17 03:21:15.363000+00:00,Modified,CWE-120,[CWE-120],python,pillow,"[python, canonical, debian, fedoraproject]","[pillow, ubuntu_linux, debian_linux, fedora]",2020,2020,3.1,nvd@nist.gov,Primary,9.8,CRITICAL,NETWORK,LOW,NONE,NONE,HIGH,HIGH,HIGH
3,CVE-2020-5312,2020,2020-01-03 01:15:11.243000+00:00,2026-06-17 03:21:15.510000+00:00,Modified,CWE-120,[CWE-120],python,pillow,"[python, canonical, debian, fedoraproject]","[pillow, ubuntu_linux, debian_linux, fedora]",2020,2020,3.1,nvd@nist.gov,Primary,9.8,CRITICAL,NETWORK,LOW,NONE,NONE,HIGH,HIGH,HIGH
4,CVE-2020-5313,2020,2020-01-03 01:15:11.320000+00:00,2026-06-17 03:21:15.670000+00:00,Modified,CWE-125,[CWE-125],python,pillow,"[python, canonical, debian, fedoraproject]","[pillow, ubuntu_linux, debian_linux, fedora]",2020,2020,3.1,nvd@nist.gov,Primary,7.1,HIGH,NETWORK,LOW,NONE,REQUIRED,LOW,NONE,HIGH



Column names:
- cve
- source_file_year
- published
- last_modified
- vuln_status
- cwe
- all_cwes
- vendor
- product
- all_vendors
- all_products
- publication_year
- cve_year
- cvss_version
- cvss_source
- cvss_type
- cvss_score
- cvss_severity
- attack_vector
- attack_complexity
- privileges_required
- user_interaction
- confidentiality_impact
- integrity_impact
- availability_impact


In [9]:
# Basic Section 2 validation

print(f"Rows: {len(nvd_df):,}")
print(f"Unique CVEs: {nvd_df['cve'].nunique():,}")
print(f"Duplicate CVE rows: {nvd_df['cve'].duplicated().sum():,}")

print("\nPublication year counts:")
display(
    nvd_df["publication_year"]
    .value_counts(dropna=False)
    .sort_index()
    .to_frame("count")
)

print("\nCVSS version counts:")
display(
    nvd_df["cvss_version"]
    .value_counts(dropna=False)
    .to_frame("count")
)

Rows: 239,942
Unique CVEs: 239,942
Duplicate CVE rows: 0

Publication year counts:


,count
publication_year,
2020,14377
2021,21019
2022,25793
2023,29377
2024,40484
2025,49549
2026,59343



CVSS version counts:


,count
cvss_version,
3.1,222583
NaN,9902
4.0,5440
3.0,2007
2.0,10


## 3. Establish One Row per CVE

The master analytical dataset requires one observation per CVE.

After standardizing the NVD records, CVE uniqueness is verified before integrating additional data sources. Duplicate CVEs are not automatically removed because doing so could hide an upstream data-processing issue.

The standardized NVD dataset contains one unique row for each CVE, so no deduplication is required at this stage.

The NVD data will now serve as the base dataset onto which CISA KEV and EPSS information will be joined.

In [10]:
# Validate one-row-per-CVE structure

total_rows = len(nvd_df)
unique_cves = nvd_df["cve"].nunique()
duplicate_cves = nvd_df["cve"].duplicated().sum()
missing_cve_ids = nvd_df["cve"].isna().sum()

cve_structure_check = pd.DataFrame({
    "check": [
        "Total rows",
        "Unique CVEs",
        "Duplicate CVE rows",
        "Missing CVE identifiers"
    ],
    "value": [
        total_rows,
        unique_cves,
        duplicate_cves,
        missing_cve_ids
    ]
})

display(cve_structure_check)

,check,value
0,Total rows,239942
1,Unique CVEs,239942
2,Duplicate CVE rows,0
3,Missing CVE identifiers,0


In [11]:
# Require one valid row per CVE before continuing

if missing_cve_ids > 0:
    raise ValueError(
        f"Found {missing_cve_ids:,} records without a CVE identifier."
    )

if duplicate_cves > 0:
    raise ValueError(
        f"Found {duplicate_cves:,} duplicate CVE rows."
    )

if total_rows != unique_cves:
    raise ValueError(
        "Row count and unique CVE count do not match."
    )

print("CVE structure validated: one row per unique CVE.")

CVE structure validated: one row per unique CVE.


In [12]:
# Create the working master analytical dataframe

master_df = nvd_df.copy()

print(f"Master dataset initialized with {len(master_df):,} CVEs.")
print(f"Columns: {len(master_df.columns)}")

Master dataset initialized with 239,942 CVEs.
Columns: 25


In [13]:
# Confirm publication years are within the intended analysis period

year_check = (
    master_df["publication_year"]
    .value_counts(dropna=False)
    .sort_index()
    .rename_axis("publication_year")
    .reset_index(name="count")
)

display(year_check)

unexpected_years = master_df.loc[
    master_df["publication_year"].notna()
    & ~master_df["publication_year"].isin(YEARS),
    "publication_year"
].unique()

missing_publication_year = master_df["publication_year"].isna().sum()

print("Missing publication year:", f"{missing_publication_year:,}")
print("Unexpected publication years:", unexpected_years)

,publication_year,count
0,2020,14377
1,2021,21019
2,2022,25793
3,2023,29377
4,2024,40484
5,2025,49549
6,2026,59343


Missing publication year: 0
Unexpected publication years: []


## 4. Join CISA Known Exploited Vulnerabilities (KEV)

The CISA Known Exploited Vulnerabilities (KEV) Catalog identifies vulnerabilities with documented evidence of exploitation in the wild.

The KEV catalog is joined to the NVD dataset using the CVE identifier.

For this analysis:

- `kev_flag = True` indicates that the CVE appears in the CISA KEV catalog.
- `kev_flag = False` indicates that the CVE does not appear in the catalog.
- Absence from KEV is **not** interpreted as evidence that a vulnerability has never been exploited.

The NVD dataset remains the base population & the join does not remove CVEs that do not appear in KEV.

In [14]:
# Load CISA KEV catalog

kev_df = pd.read_csv(KEV_FILE)

print("KEV shape:", kev_df.shape)

print("\nKEV columns:")
for column in kev_df.columns:
    print("-", column)

display(kev_df.head())

KEV shape: (1694, 12)

KEV columns:
- cveID
- vendorProject
- product
- vulnerabilityName
- dateAdded
- shortDescription
- requiredAction
- dueDate
- knownRansomwareCampaignUse
- forensicTriage
- notes
- cwes


,cveID,vendorProject,product,vulnerabilityName,dateAdded,shortDescription,requiredAction,dueDate,knownRansomwareCampaignUse,forensicTriage,notes,cwes
0,CVE-2026-59822,BerriAI,LiteLLM,BerriAI LiteLLM Improper Authentication Vulner...,2026-09-02,BerriAI LiteLLM contains an improper authentic...,Apply mitigations in accordance with vendor in...,2026-09-16,Unknown,No,https://github.com/BerriAI/litellm/security/ad...,"CWE-287, CWE-306"
1,CVE-2026-48710,Kludex,Starlette,Kludex Starlette HTTP Request/Response Smuggli...,2026-09-02,Kludex Starlette contains a HTTP request/respo...,Apply mitigations in accordance with vendor in...,2026-09-16,Unknown,No,This vulnerability affects an open-source comp...,CWE-444
2,CVE-2026-49869,Kestra,Kestra OSS,Kestra OSS OS Command Injection Vulnerability,2026-09-02,Kestra OSS contains an OS command injection vu...,Apply mitigations in accordance with vendor in...,2026-09-05,Unknown,Yes,This vulnerability affects an open-source comp...,"CWE-78, CWE-184, CWE-287, CWE-918"
3,CVE-2026-82329,JFrog,Artifactory,JFrog Artifactory Improper Authentication Vuln...,2026-09-02,JFrog Artifactory contains an improper authent...,Apply mitigations in accordance with vendor in...,2026-09-05,Unknown,Yes,https://docs.jfrog.com/releases/docs/jfrog-sec...,CWE-287
4,CVE-2026-9586,Sangoma,Switchvox,Sangoma Switchvox SQL Injection Vulnerability,2026-09-02,Sangoma Switchvox contains a SQL injection vul...,Apply mitigations in accordance with vendor in...,2026-09-05,Unknown,Yes,https://sangomakb.atlassian.net/wiki/spaces/Sw...,CWE-89


In [16]:
# Prepare CISA KEV data for integration

kev_clean = kev_df[
    [
        "cveID",
        "vendorProject",
        "product",
        "dateAdded",
        "knownRansomwareCampaignUse"
    ]
].copy()

# Standardize column names
kev_clean = kev_clean.rename(
    columns={
        "cveID": "cve",
        "vendorProject": "kev_vendor",
        "product": "kev_product",
        "dateAdded": "kev_date_added",
        "knownRansomwareCampaignUse": "kev_ransomware_use"
    }
)

# Standardize CVE identifiers
kev_clean["cve"] = (
    kev_clean["cve"]
    .astype(str)
    .str.strip()
    .str.upper()
)

# Convert KEV date to datetime
kev_clean["kev_date_added"] = pd.to_datetime(
    kev_clean["kev_date_added"],
    errors="coerce"
)

print("KEV records:", f"{len(kev_clean):,}")
print("Unique KEV CVEs:", f"{kev_clean['cve'].nunique():,}")
print("Duplicate KEV CVEs:", f"{kev_clean['cve'].duplicated().sum():,}")
print(
    "Missing KEV dates:",
    f"{kev_clean['kev_date_added'].isna().sum():,}"
)

display(kev_clean.head())

KEV records: 1,694
Unique KEV CVEs: 1,694
Duplicate KEV CVEs: 0
Missing KEV dates: 0


,cve,kev_vendor,kev_product,kev_date_added,kev_ransomware_use
0,CVE-2026-59822,BerriAI,LiteLLM,2026-09-02,Unknown
1,CVE-2026-48710,Kludex,Starlette,2026-09-02,Unknown
2,CVE-2026-49869,Kestra,Kestra OSS,2026-09-02,Unknown
3,CVE-2026-82329,JFrog,Artifactory,2026-09-02,Unknown
4,CVE-2026-9586,Sangoma,Switchvox,2026-09-02,Unknown


In [17]:
# Every record in this table is a known-exploited vulnerability

kev_clean["kev_flag"] = True

In [18]:

# Join KEV to master analytical dataset

rows_before_kev = len(master_df)

master_df = master_df.merge(
    kev_clean,
    on="cve",
    how="left",
    validate="one_to_one"
)

# CVEs not matched to KEV are coded False
master_df["kev_flag"] = (
    master_df["kev_flag"]
    .fillna(False)
    .astype(bool)
)

rows_after_kev = len(master_df)

print("Rows before KEV join:", f"{rows_before_kev:,}")
print("Rows after KEV join:", f"{rows_after_kev:,}")
print("KEV matches:", f"{master_df['kev_flag'].sum():,}")

Rows before KEV join: 239,942
Rows after KEV join: 239,942
KEV matches: 1,139


In [19]:
# KEV integration summary

kev_matches = int(master_df["kev_flag"].sum())
kev_not_matched = len(master_df) - kev_matches
kev_match_rate = kev_matches / len(master_df) * 100

print(f"Total NVD CVEs: {len(master_df):,}")
print(f"Matched to KEV: {kev_matches:,}")
print(f"Not matched to KEV: {kev_not_matched:,}")
print(f"Percent matched to KEV: {kev_match_rate:.2f}%")

Total NVD CVEs: 239,942
Matched to KEV: 1,139
Not matched to KEV: 238,803
Percent matched to KEV: 0.47%


In [20]:
# Identify KEV records that are outside the NVD analytical population

nvd_cve_set = set(master_df["cve"])
kev_cve_set = set(kev_clean["cve"])

kev_matched_set = kev_cve_set & nvd_cve_set
kev_unmatched_set = kev_cve_set - nvd_cve_set

print("KEV catalog CVEs:", f"{len(kev_cve_set):,}")
print("KEV CVEs matched to NVD population:", f"{len(kev_matched_set):,}")
print("KEV CVEs outside NVD population:", f"{len(kev_unmatched_set):,}")

KEV catalog CVEs: 1,694
KEV CVEs matched to NVD population: 1,139
KEV CVEs outside NVD population: 555


In [21]:
kev_columns = [
    "cve",
    "publication_year",
    "cvss_score",
    "cvss_severity",
    "kev_flag",
    "kev_date_added",
    "kev_vendor",
    "kev_product",
    "kev_ransomware_use"
]

display(
    master_df.loc[
        master_df["kev_flag"],
        kev_columns
    ].head(10)
)

,cve,publication_year,cvss_score,cvss_severity,kev_flag,kev_date_added,kev_vendor,kev_product,kev_ransomware_use
120,CVE-2020-0601,2020,8.1,HIGH,True,2021-11-03,Microsoft,Windows,Unknown
154,CVE-2020-0638,2020,7.8,HIGH,True,2022-05-23,Microsoft,Update Notification Manager,Known
161,CVE-2020-0646,2020,9.8,CRITICAL,True,2021-11-03,Microsoft,.NET Framework,Unknown
221,CVE-2020-2551,2020,9.8,CRITICAL,True,2023-11-16,Oracle,Fusion Middleware,Unknown
223,CVE-2020-2555,2020,9.8,CRITICAL,True,2021-11-03,Oracle,Multiple Products,Unknown
558,CVE-2020-7247,2020,9.8,CRITICAL,True,2022-03-25,OpenBSD,OpenSMTPD,Unknown
622,CVE-2020-8515,2020,9.8,CRITICAL,True,2021-11-03,DrayTek,Multiple Vigor Routers,Unknown
692,CVE-2020-3118,2020,8.8,HIGH,True,2021-11-03,Cisco,IOS XR,Unknown
699,CVE-2020-8644,2020,9.8,CRITICAL,True,2021-11-03,PlaySMS,PlaySMS,Unknown
722,CVE-2020-8657,2020,9.8,CRITICAL,True,2021-11-03,EyesOfNetwork,EyesOfNetwork,Unknown


## 5. Join Exploit Prediction Scoring System (EPSS)

The Exploit Prediction Scoring System (EPSS) provides an estimate of the probability that a published CVE will be exploited.

The EPSS snapshot used in this project is dated **September 2, 2026**.

Two EPSS measures will be added to the master analytical dataset:

- **EPSS score:** estimated probability of exploitation
- **EPSS percentile:** relative ranking of the vulnerability compared with other scored CVEs

Because this project uses a current EPSS snapshot, EPSS is treated as a cross-sectional measure. The score should not be interpreted as the EPSS value that existed when an older vulnerability was originally published or exploited.

The NVD dataset remains the base analytical population, so the EPSS join must preserve all existing CVE rows.

In [22]:
# Load EPSS snapshot

epss_df = pd.read_csv(
    EPSS_FILE,
    compression="gzip",
    comment="#"
)

print("EPSS shape:", epss_df.shape)

print("\nEPSS columns:")
for column in epss_df.columns:
    print("-", column)

display(epss_df.head())

EPSS shape: (367327, 3)

EPSS columns:
- cve
- epss
- percentile


,cve,epss,percentile
0,CVE-1999-0001,0.03351,0.87851
1,CVE-1999-0002,0.27858,0.97956
2,CVE-1999-0003,0.24565,0.97723
3,CVE-1999-0004,0.02817,0.85584
4,CVE-1999-0005,0.18415,0.97030


In [23]:
# ---------------------------------------------------------
# Prepare EPSS data for integration
# ---------------------------------------------------------

epss_clean = epss_df[
    ["cve", "epss", "percentile"]
].copy()

# Standardize CVE identifiers
epss_clean["cve"] = (
    epss_clean["cve"]
    .astype(str)
    .str.strip()
    .str.upper()
)

# Rename analytical fields
epss_clean = epss_clean.rename(
    columns={
        "epss": "epss_score",
        "percentile": "epss_percentile"
    }
)

# Ensure scores are numeric
epss_clean["epss_score"] = pd.to_numeric(
    epss_clean["epss_score"],
    errors="coerce"
)

epss_clean["epss_percentile"] = pd.to_numeric(
    epss_clean["epss_percentile"],
    errors="coerce"
)

print("EPSS records:", f"{len(epss_clean):,}")
print("Unique EPSS CVEs:", f"{epss_clean['cve'].nunique():,}")
print("Duplicate EPSS CVEs:", f"{epss_clean['cve'].duplicated().sum():,}")
print("Missing EPSS scores:", f"{epss_clean['epss_score'].isna().sum():,}")
print(
    "Missing EPSS percentiles:",
    f"{epss_clean['epss_percentile'].isna().sum():,}"
)

EPSS records: 367,327
Unique EPSS CVEs: 367,327
Duplicate EPSS CVEs: 0
Missing EPSS scores: 0
Missing EPSS percentiles: 0


In [24]:
# Join EPSS to master analytical dataset

rows_before_epss = len(master_df)

master_df = master_df.merge(
    epss_clean,
    on="cve",
    how="left",
    validate="one_to_one"
)

rows_after_epss = len(master_df)

print("Rows before EPSS join:", f"{rows_before_epss:,}")
print("Rows after EPSS join:", f"{rows_after_epss:,}")

Rows before EPSS join: 239,942
Rows after EPSS join: 239,942


In [25]:
# EPSS integration summary

epss_matches = master_df["epss_score"].notna().sum()
epss_not_matched = master_df["epss_score"].isna().sum()
epss_match_rate = epss_matches / len(master_df) * 100

print(f"Total NVD CVEs: {len(master_df):,}")
print(f"Matched to EPSS: {epss_matches:,}")
print(f"Not matched to EPSS: {epss_not_matched:,}")
print(f"Percent matched to EPSS: {epss_match_rate:.2f}%")

Total NVD CVEs: 239,942
Matched to EPSS: 232,116
Not matched to EPSS: 7,826
Percent matched to EPSS: 96.74%


In [26]:
# Validate EPSS score ranges

invalid_epss_scores = master_df.loc[
    master_df["epss_score"].notna()
    & ~master_df["epss_score"].between(0, 1)
]

invalid_epss_percentiles = master_df.loc[
    master_df["epss_percentile"].notna()
    & ~master_df["epss_percentile"].between(0, 1)
]

print(
    "EPSS scores outside 0–1:",
    f"{len(invalid_epss_scores):,}"
)

print(
    "EPSS percentiles outside 0–1:",
    f"{len(invalid_epss_percentiles):,}"
)

print("\nEPSS score range:")
print(
    master_df["epss_score"].min(),
    "to",
    master_df["epss_score"].max()
)

print("\nEPSS percentile range:")
print(
    master_df["epss_percentile"].min(),
    "to",
    master_df["epss_percentile"].max()
)

EPSS scores outside 0–1: 0
EPSS percentiles outside 0–1: 0

EPSS score range:
0.00047 to 0.99999

EPSS percentile range:
0.0 to 1.0


In [27]:
# Check EPSS availability among known-exploited vulnerabilities

kev_epss_summary = (
    master_df
    .groupby("kev_flag", observed=True)
    .agg(
        total_cves=("cve", "size"),
        epss_available=("epss_score", "count")
    )
    .reset_index()
)

kev_epss_summary["epss_coverage_pct"] = (
    kev_epss_summary["epss_available"]
    / kev_epss_summary["total_cves"]
    * 100
)

display(kev_epss_summary)

,kev_flag,total_cves,epss_available,epss_coverage_pct
0,False,238803,230977,96.722822
1,True,1139,1139,100.000000


In [28]:
display_columns = [
    "cve",
    "publication_year",
    "cvss_score",
    "cvss_severity",
    "kev_flag",
    "epss_score",
    "epss_percentile"
]

display(
    master_df[display_columns].head(10)
)

print("\nMaster dataset shape:", master_df.shape)

,cve,publication_year,cvss_score,cvss_severity,kev_flag,epss_score,epss_percentile
0,CVE-2020-5179,2020,7.2,HIGH,False,0.02840,0.85704
1,CVE-2020-5310,2020,8.8,HIGH,False,0.01975,0.79083
2,CVE-2020-5311,2020,9.8,CRITICAL,False,0.04212,0.90288
3,CVE-2020-5312,2020,9.8,CRITICAL,False,0.03690,0.88935
4,CVE-2020-5313,2020,7.1,HIGH,False,0.02752,0.85202
5,CVE-2020-1785,2020,5.5,MEDIUM,False,0.00478,0.39450
6,CVE-2020-1871,2020,8.2,HIGH,False,0.00644,0.48472
7,CVE-2020-5395,2020,8.8,HIGH,False,0.02478,0.83450
8,CVE-2020-5496,2020,8.8,HIGH,False,0.02365,0.82637
9,CVE-2020-5497,2020,6.1,MEDIUM,False,0.02133,0.80726



Master dataset shape: (239942, 32)


## 6. Create CVSS Threshold and KEV Mismatch Features

This section creates analysis-ready features representing hypothetical CVSS-based prioritization thresholds.

Three thresholds are evaluated:

- **Below 7.0:** CVEs that would fall below a threshold prioritizing High and Critical vulnerabilities.
- **Below 8.0:** CVEs with CVSS scores below 8.0.
- **Below 9.0:** CVEs that would fall below a threshold prioritizing only Critical vulnerabilities.

For each threshold, a corresponding **KEV mismatch indicator** identifies vulnerabilities that:

1. Are listed in the CISA Known Exploited Vulnerabilities (KEV) Catalog, and
2. Have a CVSS score below the specified prioritization threshold.

For example, `kev_mismatch_9 = True` means that a vulnerability is known to be exploited but has a CVSS score below 9.0.

These indicators are used to measure how often severity-only prioritization could under-prioritize vulnerabilities with documented real-world exploitation.

CVEs without a CVSS score are not classified as being above or below a threshold. Their threshold and mismatch indicators are therefore stored as missing rather than `False`.

In [29]:
# Create CVSS threshold indicators

thresholds = [7, 8, 9]

for threshold in thresholds:

    threshold_col = f"below_{threshold}"

    master_df[threshold_col] = pd.Series(
        pd.NA,
        index=master_df.index,
        dtype="boolean"
    )

    has_cvss = master_df["cvss_score"].notna()

    master_df.loc[
        has_cvss,
        threshold_col
    ] = (
        master_df.loc[has_cvss, "cvss_score"]
        < threshold
    )


threshold_columns = [
    "below_7",
    "below_8",
    "below_9"
]

display(
    master_df[
        ["cve", "cvss_score"] + threshold_columns
    ].head(15)
)

,cve,cvss_score,below_7,below_8,below_9
0,CVE-2020-5179,7.2,False,True,True
1,CVE-2020-5310,8.8,False,False,True
2,CVE-2020-5311,9.8,False,False,False
3,CVE-2020-5312,9.8,False,False,False
4,CVE-2020-5313,7.1,False,True,True
5,CVE-2020-1785,5.5,True,True,True
6,CVE-2020-1871,8.2,False,False,True
7,CVE-2020-5395,8.8,False,False,True
8,CVE-2020-5496,8.8,False,False,True
9,CVE-2020-5497,6.1,True,True,True


In [31]:
# Create KEV mismatch indicators

for threshold in thresholds:

    threshold_col = f"below_{threshold}"
    mismatch_col = f"kev_mismatch_{threshold}"

    master_df[mismatch_col] = pd.Series(
        pd.NA,
        index=master_df.index,
        dtype="boolean"
    )

    has_cvss = master_df["cvss_score"].notna()

    master_df.loc[
        has_cvss,
        mismatch_col
    ] = (
        master_df.loc[has_cvss, "kev_flag"]
        & master_df.loc[has_cvss, threshold_col]
    )


mismatch_columns = [
    "kev_mismatch_7",
    "kev_mismatch_8",
    "kev_mismatch_9"
]

In [32]:
# Validate threshold boundaries

boundary_check = (
    master_df.loc[
        master_df["cvss_score"].isin(
            [6.9, 7.0, 7.9, 8.0, 8.9, 9.0]
        ),
        [
            "cve",
            "cvss_score",
            "below_7",
            "below_8",
            "below_9"
        ]
    ]
    .sort_values("cvss_score")
    .groupby("cvss_score")
    .head(1)
)

display(boundary_check)

,cve,cvss_score,below_7,below_8,below_9
239845,CVE-2026-84481,6.9,True,True,True
163313,CVE-2025-0712,7.0,False,True,True
175754,CVE-2025-30185,7.9,False,True,True
196767,CVE-2026-33335,8.0,False,False,True
192129,CVE-2026-27169,8.9,False,False,True
9462,CVE-2020-15179,9.0,False,False,False


In [33]:
# Summarize KEV mismatches across CVSS thresholds

kev_with_cvss = master_df.loc[
    master_df["kev_flag"]
    & master_df["cvss_score"].notna()
].copy()

mismatch_summary = []

for threshold in thresholds:

    mismatch_col = f"kev_mismatch_{threshold}"

    mismatch_count = int(
        kev_with_cvss[mismatch_col].sum()
    )

    total_kev_with_cvss = len(kev_with_cvss)

    mismatch_pct = (
        mismatch_count
        / total_kev_with_cvss
        * 100
    )

    mismatch_summary.append({
        "cvss_threshold": threshold,
        "kev_with_cvss": total_kev_with_cvss,
        "below_threshold": mismatch_count,
        "percent_below_threshold": mismatch_pct
    })


mismatch_summary_df = pd.DataFrame(
    mismatch_summary
)

display(mismatch_summary_df)

,cvss_threshold,kev_with_cvss,below_threshold,percent_below_threshold
0,7,1138,138,12.126538
1,8,1138,474,41.652021
2,9,1138,694,60.984183


In [34]:
# Confirm that missing CVSS values remain missing
# in threshold and mismatch features

missing_cvss_check = master_df.loc[
    master_df["cvss_score"].isna(),
    [
        "cve",
        "cvss_score",
        "kev_flag",
        "below_7",
        "below_8",
        "below_9",
        "kev_mismatch_7",
        "kev_mismatch_8",
        "kev_mismatch_9"
    ]
]

print(
    "CVEs without CVSS score:",
    f"{len(missing_cvss_check):,}"
)

display(missing_cvss_check.head())

CVEs without CVSS score: 9,902


,cve,cvss_score,kev_flag,below_7,below_8,below_9,kev_mismatch_7,kev_mismatch_8,kev_mismatch_9
438,CVE-2020-5498,NaN,False,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
1142,CVE-2020-8998,NaN,False,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
1400,CVE-2020-6764,NaN,False,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
1483,CVE-2020-7975,NaN,False,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
1502,CVE-2020-10020,NaN,False,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>


## 7. Data Quality Assurance

Before exporting the master analytical dataset, I performed a final set of quality-assurance checks.

The QA process evaluates:

- Total observations and unique CVEs
- Duplicate and missing CVE identifiers
- Publication-year coverage
- CVSS availability
- CVSS v3.x coverage
- CISA KEV matches
- EPSS matches
- Missingness in key analytical variables
- Validity of CVSS and EPSS score ranges
- Integrity of the threshold and mismatch features

These checks ensure that the processed dataset is internally consistent and suitable for downstream exploratory and statistical analysis.

In [35]:
# Master dataset QA summary

total_rows = len(master_df)
unique_cves = master_df["cve"].nunique()
duplicate_cves = master_df["cve"].duplicated().sum()
missing_cves = master_df["cve"].isna().sum()

cvss_available = master_df["cvss_score"].notna().sum()

cvss_v3x = (
    master_df["cvss_version"]
    .astype(str)
    .isin(["3.0", "3.1"])
    .sum()
)

kev_matches = master_df["kev_flag"].sum()
epss_matches = master_df["epss_score"].notna().sum()

qa_summary = pd.DataFrame({
    "metric": [
        "Total rows",
        "Unique CVEs",
        "Duplicate CVE rows",
        "Missing CVE identifiers",
        "CVSS scores available",
        "CVSS v3.x records",
        "KEV matches",
        "EPSS matches"
    ],
    "count": [
        total_rows,
        unique_cves,
        duplicate_cves,
        missing_cves,
        cvss_available,
        cvss_v3x,
        kev_matches,
        epss_matches
    ]
})

qa_summary["percent_of_dataset"] = (
    qa_summary["count"]
    / total_rows
    * 100
)

display(qa_summary)

,metric,count,percent_of_dataset
0,Total rows,239942,100.000000
1,Unique CVEs,239942,100.000000
2,Duplicate CVE rows,0,0.000000
3,Missing CVE identifiers,0,0.000000
4,CVSS scores available,230040,95.873169
5,CVSS v3.x records,224590,93.601787
6,KEV matches,1139,0.474698
7,EPSS matches,232116,96.738378


In [36]:
# Missingness in key analytical variables

qa_columns = [
    "cve",
    "publication_year",
    "cvss_score",
    "cvss_severity",
    "cvss_version",
    "attack_vector",
    "attack_complexity",
    "privileges_required",
    "user_interaction",
    "cwe",
    "vendor",
    "product",
    "epss_score",
    "epss_percentile"
]

missingness_qa = pd.DataFrame({
    "variable": qa_columns,
    "missing_count": [
        master_df[col].isna().sum()
        for col in qa_columns
    ]
})

missingness_qa["missing_percent"] = (
    missingness_qa["missing_count"]
    / len(master_df)
    * 100
)

missingness_qa = missingness_qa.sort_values(
    "missing_percent",
    ascending=False
).reset_index(drop=True)

display(missingness_qa)

,variable,missing_count,missing_percent
0,product,60680,25.289445
1,vendor,60679,25.289028
2,cwe,28324,11.804519
3,attack_vector,9912,4.130998
4,attack_complexity,9912,4.130998
5,privileges_required,9912,4.130998
6,user_interaction,9912,4.130998
7,cvss_score,9902,4.126831
8,cvss_severity,9902,4.126831
9,cvss_version,9902,4.126831


In [37]:
# Validate analytical score ranges

range_qa = pd.DataFrame({
    "variable": [
        "CVSS score",
        "EPSS score",
        "EPSS percentile"
    ],
    "expected_range": [
        "0–10",
        "0–1",
        "0–1"
    ],
    "observed_min": [
        master_df["cvss_score"].min(),
        master_df["epss_score"].min(),
        master_df["epss_percentile"].min()
    ],
    "observed_max": [
        master_df["cvss_score"].max(),
        master_df["epss_score"].max(),
        master_df["epss_percentile"].max()
    ],
    "invalid_count": [
        (
            master_df["cvss_score"].notna()
            & ~master_df["cvss_score"].between(0, 10)
        ).sum(),

        (
            master_df["epss_score"].notna()
            & ~master_df["epss_score"].between(0, 1)
        ).sum(),

        (
            master_df["epss_percentile"].notna()
            & ~master_df["epss_percentile"].between(0, 1)
        ).sum()
    ]
})

display(range_qa)

,variable,expected_range,observed_min,observed_max,invalid_count
0,CVSS score,0–10,0.00000,10.00000,0
1,EPSS score,0–1,0.00047,0.99999,0
2,EPSS percentile,0–1,0.00000,1.00000,0


In [38]:
# Validate threshold and mismatch feature logic

feature_checks = []

for threshold in thresholds:

    below_col = f"below_{threshold}"
    mismatch_col = f"kev_mismatch_{threshold}"

    has_cvss = master_df["cvss_score"].notna()
    missing_cvss = master_df["cvss_score"].isna()

    incorrect_threshold = (
        master_df.loc[has_cvss, below_col]
        != (
            master_df.loc[has_cvss, "cvss_score"]
            < threshold
        )
    ).sum()

    incorrect_mismatch = (
        master_df.loc[has_cvss, mismatch_col]
        != (
            master_df.loc[has_cvss, "kev_flag"]
            & master_df.loc[has_cvss, below_col]
        )
    ).sum()

    missing_threshold_errors = (
        master_df.loc[
            missing_cvss,
            below_col
        ].notna().sum()
    )

    missing_mismatch_errors = (
        master_df.loc[
            missing_cvss,
            mismatch_col
        ].notna().sum()
    )

    feature_checks.append({
        "threshold": threshold,
        "incorrect_threshold_flags": incorrect_threshold,
        "incorrect_mismatch_flags": incorrect_mismatch,
        "missing_cvss_threshold_errors": missing_threshold_errors,
        "missing_cvss_mismatch_errors": missing_mismatch_errors
    })


feature_qa = pd.DataFrame(feature_checks)

display(feature_qa)

,threshold,incorrect_threshold_flags,incorrect_mismatch_flags,missing_cvss_threshold_errors,missing_cvss_mismatch_errors
0,7,0,0,0,0
1,8,0,0,0,0
2,9,0,0,0,0


In [39]:
# Final pipeline integrity checks

assert len(master_df) == 239_942, \
    "Unexpected master dataset row count."

assert master_df["cve"].nunique() == len(master_df), \
    "CVE identifiers are not unique."

assert master_df["cve"].isna().sum() == 0, \
    "Missing CVE identifiers detected."

assert master_df["publication_year"].isna().sum() == 0, \
    "Missing publication years detected."

assert master_df["publication_year"].isin(YEARS).all(), \
    "Unexpected publication year detected."

assert (
    range_qa["invalid_count"] == 0
).all(), "Invalid analytical score detected."

assert (
    feature_qa.drop(columns="threshold") == 0
).all().all(), "Feature validation failed."

print("All final data QA checks passed.")

All final data QA checks passed.


## 8. Export Master Analytical Dataset

After completing the integration and quality-assurance checks, the master analytical dataset is exported for use in downstream exploratory analysis, statistical modeling, visualization, and dashboard development.

The processed dataset contains one row per CVE and combines:

- NVD vulnerability and CVSS information
- CISA KEV known-exploitation information
- EPSS exploitation probability and percentile
- CVSS threshold indicators
- KEV mismatch indicators

The primary processed file is stored in Parquet format to preserve data types and provide efficient storage and loading.

The exported file is:

`data/processed/master_vulnerability_dataset.csv`

Future analytical notebooks should load this processed dataset rather than repeatedly parsing the raw NVD JSON files.

In [40]:
# Organize final master dataset columns

final_columns = [
    # Identification and time
    "cve",
    "published",
    "last_modified",
    "publication_year",
    "cve_year",
    "source_file_year",
    "vuln_status",

    # CVSS
    "cvss_score",
    "cvss_severity",
    "cvss_version",
    "cvss_source",
    "cvss_type",

    # Attack characteristics
    "attack_vector",
    "attack_complexity",
    "privileges_required",
    "user_interaction",
    "confidentiality_impact",
    "integrity_impact",
    "availability_impact",

    # Vulnerability classification
    "cwe",
    "all_cwes",

    # Vendor and product
    "vendor",
    "product",
    "all_vendors",
    "all_products",

    # CISA KEV
    "kev_flag",
    "kev_date_added",
    "kev_vendor",
    "kev_product",
    "kev_ransomware_use",

    # EPSS
    "epss_score",
    "epss_percentile",

    # CVSS threshold features
    "below_7",
    "below_8",
    "below_9",

    # KEV mismatch features
    "kev_mismatch_7",
    "kev_mismatch_8",
    "kev_mismatch_9"
]

master_df = master_df[final_columns].copy()

print("Final master dataset shape:", master_df.shape)

Final master dataset shape: (239942, 38)


In [42]:
# Export master analytical dataset

master_df.to_csv(
    MASTER_DATA_FILE,
    index=False
)

print("Master dataset exported successfully.")
print("File:", MASTER_DATA_FILE)
print("Rows:", f"{len(master_df):,}")
print("Columns:", len(master_df.columns))

Master dataset exported successfully.
File: /Users/aaliyahmjh/Documents/data698-capstone/data/processed/master_vulnerability_dataset.csv
Rows: 239,942
Columns: 38


In [43]:
# Verify exported dataset

reloaded_df = pd.read_csv(
    MASTER_DATA_FILE
)

print("Exported file exists:", MASTER_DATA_FILE.exists())
print("Reloaded shape:", reloaded_df.shape)

assert reloaded_df.shape == master_df.shape, \
    "Reloaded dataset shape does not match master dataset."

assert reloaded_df["cve"].nunique() == len(reloaded_df), \
    "Reloaded dataset does not contain one row per CVE."

assert reloaded_df["cve"].isna().sum() == 0, \
    "Reloaded dataset contains missing CVE identifiers."

assert set(reloaded_df.columns) == set(master_df.columns), \
    "Reloaded dataset columns do not match master dataset."

print("\nProcessed dataset successfully reloaded and validated.")

Exported file exists: True
Reloaded shape: (239942, 38)

Processed dataset successfully reloaded and validated.


## Data Preparation Complete

The master analytical dataset has been successfully constructed and validated.

### Final Dataset

- **Unit of analysis:** One row per CVE
- **Study period:** 2020–2026
- **Sources:** NVD, CISA KEV, and EPSS
- **Output:** `data/processed/master_vulnerability_dataset.csv`

The dataset includes CVSS severity and attack characteristics, vulnerability classifications, vendor/product information, known-exploitation status, EPSS scores, and derived CVSS threshold/mismatch features.

All final QA checks passed before export.

This processed dataset will serve as the authoritative input for subsequent exploratory analysis, statistical testing, modeling, visualization, and dashboard development.